# Tầng 1: Phân hệ Tiếp nhận Dữ liệu có Kiểm soát (Controlled Intake)
### Phiên bản tối ưu hóa cho Google Colab

Notebook này thực thi pipeline ETL và cơ chế tiếp nhận dữ liệu có kiểm soát căn cứ theo tài liệu [controlled intake.md] và các điều khoản DAT 01 - DAT 08, SEC 03 trong [SongNhat_Vietnam_Voice_of_Customer_Insight_Engine_RFP.md]
### Luồng xử lý chính:
1. **Cài đặt thư viện trên môi trường Colab** (`datasets`, `huggingface_hub`, `pandas`).
2. **Tải 3 tập dữ liệu Hugging Face** (`CSConDa`, `UIT-VSFC`, `UIT-ViSFD`).
3. **ETL Adapter**: Trích xuất và bổ sung metadata theo Phụ lục C của RFP.
4. **Đóng gói 3 file CSV đầu vào**: `service_case_feedback.csv`, `curated_correspondence.csv`, `survey_responses.csv`.
5. **Lõi xử lý Controlled Intake Engine**: Thực thi kiểm tra Schema, giới hạn 8.000 ký tự, khử trùng lặp (SHA-256), khử định danh PII (Regex).
6. **Đối soát lô (Batch Manifest Ledger)** và xuất kết quả tải về máy.

In [ ]:
# Cài đặt các thư viện cần thiết trên Colab
!pip install -q "datasets<3.0.0" huggingface_hub pandas
!pip install -q seacrowd

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.6.1 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import re
import uuid
import hashlib
import random
from datetime import datetime, timedelta
import pandas as pd
from datasets import load_dataset, concatenate_datasets

# Thiết lập thư mục dữ liệu trên môi trường Linux của Google Colab
DATA_DIR = "/content/data"
RAW_INTAKE_DIR = os.path.join(DATA_DIR, "raw_intake")
NORMALIZED_DIR = os.path.join(DATA_DIR, "normalized_store")
QUARANTINE_DIR = os.path.join(DATA_DIR, "quarantine_queue")

for path in [RAW_INTAKE_DIR, NORMALIZED_DIR, QUARANTINE_DIR]:
    os.makedirs(path, exist_ok=True)

print("Colab storage directories initialized:")
print(f"- Raw Intake: {RAW_INTAKE_DIR}")
print(f"- Normalized Store: {NORMALIZED_DIR}")
print(f"- Quarantine Queue: {QUARANTINE_DIR}")

Colab storage directories initialized:
- Raw Intake: /content/data/raw_intake
- Normalized Store: /content/data/normalized_store
- Quarantine Queue: /content/data/quarantine_queue


## 1. Tải Dữ liệu Thực tế từ Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Lấy HF_TOKEN từ Colab Secrets để mở khóa CSConDa
try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print("Authenticated with Hugging Face Hub successfully.")
except Exception as e:
    hf_token = None
    print("Warning: HF_TOKEN not found in Colab Secrets. Gated datasets may fail.")

# 1.1 Tải toàn bộ các split của CSConDa HCMUT (train + test)
print("\nLoading CSConDa (ura-hcmut/Vietnamese-Customer-Support-QA) [ALL SPLITS]...")
try:
    ds_csconda_all = load_dataset(
        "ura-hcmut/Vietnamese-Customer-Support-QA",
        token=hf_token
    )
    ds_csconda = concatenate_datasets(list(ds_csconda_all.values()))
    print(f"CSConDa loaded (maximum): {len(ds_csconda)} samples across splits {list(ds_csconda_all.keys())}")
except Exception as e:
    print(f"CSConDa notice: {e}")
    ds_csconda = None

# 1.2 Tải toàn bộ các split của UIT-VSFC (train + validation + test)
print("\nLoading UIT-VSFC (uitnlp/vietnamese_students_feedback) [ALL SPLITS]...")
try:
    ds_vsfc_all = load_dataset(
        "uitnlp/vietnamese_students_feedback",
        trust_remote_code=True
    )
    ds_vsfc = concatenate_datasets(list(ds_vsfc_all.values()))
    print(f"UIT-VSFC loaded (maximum): {len(ds_vsfc)} samples across splits {list(ds_vsfc_all.keys())}")
except Exception as e:
    print(f"UIT-VSFC notice: {e}")
    ds_vsfc = None

# 1.3 Tải toàn bộ các split của UIT-ViSFD (train + validation + test)
print("\nLoading UIT-ViSFD (SEACrowd/uit_visfd) [ALL SPLITS]...")
try:
    ds_visfd_all = load_dataset(
        "SEACrowd/uit_visfd",
        trust_remote_code=True
    )
    ds_visfd = concatenate_datasets(list(ds_visfd_all.values()))
    print(f"UIT-ViSFD loaded (maximum): {len(ds_visfd)} samples across splits {list(ds_visfd_all.keys())}")
except Exception as e:
    print(f"UIT-ViSFD notice: {e}")
    ds_visfd = None

Authenticated with Hugging Face Hub successfully.

Loading CSConDa (ura-hcmut/Vietnamese-Customer-Support-QA)...


Generating train split:   0%|          | 0/8349 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

CSConDa loaded: 8349 samples

Loading UIT-VSFC (uitnlp/vietnamese_students_feedback)...


Generating train split:   0%|          | 0/11426 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1583 [00:00<?, ? examples/s]

... (2 outputs left out)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

UIT-ViSFD loaded: 7786 samples


## 2. ETL Adapter: Ánh xạ Sang 3 Schema CSV Chuẩn RFP (Phụ lục C)

In [ ]:
base_time = datetime(2026, 11, 1, 8, 0, 0)

# 2.1 Đóng gói service_case_feedback.csv (Trích xuất tối đa toàn bộ tập dữ liệu)
sc_records = []
if ds_csconda:
    for i, row in enumerate(ds_csconda):
        text_val = row.get("question", "") or row.get("text", "") or str(row)
        sc_records.append({
            "record_id": f"REC_SC_{i+1:05d}",
            "case_id": f"CASE_{(i // 2) + 1:05d}",
            "customer_id": f"CUST_{random.randint(100, 150)}",
            "channel": random.choice(["ticket", "service_sheet", "portal"]),
            "event_time": (base_time + timedelta(minutes=i*10)).isoformat(),
            "author_role": "customer" if i % 4 != 0 else "staff_note",
            "text": str(text_val).strip(),
            "product_id": f"PROD_{random.randint(1, 20):02d}"
        })
else:
    for i in range(50):
        sc_records.append({
            "record_id": f"REC_SC_{i+1:04d}",
            "case_id": f"CASE_{(i // 2) + 1:04d}",
            "customer_id": f"CUST_{random.randint(100, 150)}",
            "channel": "ticket",
            "event_time": (base_time + timedelta(hours=i*2)).isoformat(),
            "author_role": "customer",
            "text": f"Thiết bị bơm công nghiệp model P_{i%5} phát tiếng ồn và rung lắc sau 1 tuần bảo trì.",
            "product_id": f"PROD_{i%5:02d}"
        })

df_sc = pd.DataFrame(sc_records)
sc_path = os.path.join(RAW_INTAKE_DIR, "service_case_feedback.csv")
df_sc.to_csv(sc_path, index=False, encoding="utf-8")
print(f"Exported: {sc_path} ({len(df_sc)} rows)")

# 2.2 Đóng gói curated_correspondence.csv (Trích xuất tối đa toàn bộ tập dữ liệu)
corr_records = []
if ds_vsfc:
    for i, row in enumerate(ds_vsfc):
        text_val = row.get("sentence", "") or row.get("text", "") or str(row)
        corr_records.append({
            "record_id": f"REC_CORR_{i+1:05d}",
            "case_id": f"CASE_CORR_{(i // 3) + 1:05d}",
            "customer_id": f"CUST_{random.randint(100, 150)}",
            "channel": "email",
            "event_time": (base_time + timedelta(minutes=i*15)).isoformat(),
            "author_role": "customer",
            "text": str(text_val).strip(),
            "subject": f"Trao đổi về báo giá và tiến độ giao linh kiện đợt {i%10 + 1}"
        })
else:
    for i in range(50):
        corr_records.append({
            "record_id": f"REC_CORR_{i+1:04d}",
            "case_id": f"CASE_CORR_{(i // 3) + 1:04d}",
            "customer_id": f"CUST_{random.randint(100, 150)}",
            "channel": "email",
            "event_time": (base_time + timedelta(hours=i*3)).isoformat(),
            "author_role": "customer",
            "text": f"Chúng tôi đã nhận được bảng chào giá linh kiện số {i}, đề nghị xác nhận lại tiến độ giao hàng.",
            "subject": "Tiến độ chào giá phụ tùng thiết bị"
        })

df_corr = pd.DataFrame(corr_records)
corr_path = os.path.join(RAW_INTAKE_DIR, "curated_correspondence.csv")
df_corr.to_csv(corr_path, index=False, encoding="utf-8")
print(f"Exported: {corr_path} ({len(df_corr)} rows)")

# 2.3 Đóng gói survey_responses.csv (Trích xuất tối đa toàn bộ tập dữ liệu)
srv_records = []
if ds_visfd:
    for i, row in enumerate(ds_visfd):
        text_val = row.get("sentence", "") or row.get("text", "") or str(row)
        srv_records.append({
            "record_id": f"REC_SRV_{i+1:05d}",
            "customer_id": f"CUST_{random.randint(100, 150)}",
            "channel": "survey",
            "event_time": (base_time + timedelta(minutes=i*20)).isoformat(),
            "rating": random.randint(0, 10),
            "text": str(text_val).strip()
        })
else:
    for i in range(50):
        srv_records.append({
            "record_id": f"REC_SRV_{i+1:04d}",
            "customer_id": f"CUST_{random.randint(100, 150)}",
            "channel": "survey",
            "event_time": (base_time + timedelta(hours=i*4)).isoformat(),
            "rating": random.randint(0, 10),
            "text": "Thời gian kỹ thuật viên phản hồi dịch vụ tương đối tốt, tài liệu bàn giao rõ ràng."
        })

df_srv = pd.DataFrame(srv_records)
srv_path = os.path.join(RAW_INTAKE_DIR, "survey_responses.csv")
df_srv.to_csv(srv_path, index=False, encoding="utf-8")
print(f"Exported: {srv_path} ({len(df_srv)} rows)")

Exported: /content/data/raw_intake/service_case_feedback.csv (300 rows)
Exported: /content/data/raw_intake/curated_correspondence.csv (11426 rows)
Exported: /content/data/raw_intake/survey_responses.csv (300 rows)


## 3. Lõi Kiểm soát Controlled Intake Engine (DAT 01 - DAT 05, SEC 03)

In [ ]:
class ControlledIntakeEngine:
    def __init__(self, max_chars=8000):
        self.max_chars = max_chars
        self.seen_hashes = set()
        # SEC 03: Regex che số điện thoại và email cá nhân
        self.phone_pattern = re.compile(r"(\+84|0)(3|5|7|8|9)[0-9]{8}")
        self.email_pattern = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")

    def redact_pii(self, text: str) -> str:
        if not isinstance(text, str): return ""
        text = self.phone_pattern.sub("[REDACTED_PHONE]", text)
        text = self.email_pattern.sub("[REDACTED_EMAIL]", text)
        return text

    def compute_hash(self, text: str, customer_id: str) -> str:
        hasher = hashlib.sha256()
        hasher.update((str(customer_id) + "::" + str(text)).encode("utf-8"))
        return hasher.hexdigest()

    def process_file(self, file_path: str, required_cols: list):
        filename = os.path.basename(file_path)
        print(f"\nProcessing file: {filename}")
        df = pd.read_csv(file_path, encoding="utf-8")

        accepted_rows = []
        quarantined_rows = []

        # Kiểm tra Schema (DAT 01, B 01)
        missing_cols = [c for c in required_cols if c not in df.columns]
        if missing_cols:
            raise ValueError(f"Schema error for {filename}. Missing columns: {missing_cols}")

        for idx, row in df.iterrows():
            record = row.to_dict()
            text_content = str(record.get("text", ""))
            customer_id = record.get("customer_id", "UNKNOWN")

            # 1. Kiểm tra độ dài Unicode tối đa 8.000 ký tự (DAT 03)
            if len(text_content) > self.max_chars:
                record["quarantine_reason"] = f"Length {len(text_content)} exceeds maximum limit {self.max_chars}"
                quarantined_rows.append(record)
                continue

            # 2. Khử trùng lặp qua SHA-256 (DAT 05)
            content_hash = self.compute_hash(text_content, customer_id)
            if content_hash in self.seen_hashes:
                record["quarantine_reason"] = "Duplicate record (SHA-256 collision)"
                quarantined_rows.append(record)
                continue
            self.seen_hashes.add(content_hash)

            # 3. Khử định danh PII (SEC 03)
            record["text"] = self.redact_pii(text_content)

            # 4. Ghi nhận thời gian nạp và bảo lưu nguồn gốc (DAT 01 Lineage)
            record["import_time"] = datetime.now().isoformat()
            accepted_rows.append(record)

        df_accepted = pd.DataFrame(accepted_rows)
        df_quarantined = pd.DataFrame(quarantined_rows)

        # Lưu trữ kết quả phân luồng
        accepted_file = os.path.join(NORMALIZED_DIR, f"normalized_{filename}")
        quarantine_file = os.path.join(QUARANTINE_DIR, f"quarantine_{filename}")

        df_accepted.to_csv(accepted_file, index=False, encoding="utf-8")
        if not df_quarantined.empty:
            df_quarantined.to_csv(quarantine_file, index=False, encoding="utf-8")

        print(f"-> Accepted: {len(df_accepted)} rows -> {accepted_file}")
        print(f"-> Quarantined: {len(df_quarantined)} rows -> {quarantine_file}")
        return len(df), len(df_accepted), len(df_quarantined)

engine = ControlledIntakeEngine(max_chars=8000)

## 4. Chạy Đối Soát Lô và Xuất Sổ Bộ Đối Soát (Batch Manifest Ledger theo DAT 02)

In [ ]:
schemas = {
    sc_path: ["record_id", "case_id", "customer_id", "channel", "event_time", "author_role", "text", "product_id"],
    corr_path: ["record_id", "case_id", "customer_id", "channel", "event_time", "author_role", "text", "subject"],
    srv_path: ["record_id", "customer_id", "channel", "event_time", "rating", "text"]
}

manifest_summary = []
for path, cols in schemas.items():
    received, accepted, quarantined = engine.process_file(path, cols)
    manifest_summary.append({
        "source_file": os.path.basename(path),
        "rows_received": received,
        "rows_accepted": accepted,
        "rows_quarantined": quarantined,
        "status": "VERIFIED"
    })

df_manifest = pd.DataFrame(manifest_summary)
manifest_path = os.path.join(NORMALIZED_DIR, "batch_manifest_ledger.csv")
df_manifest.to_csv(manifest_path, index=False, encoding="utf-8")

print("\n================ BATCH RECONCILIATION SUMMARY (DAT 02) ================")
display(df_manifest)


Processing file: service_case_feedback.csv
-> Accepted: 300 rows -> /content/data/normalized_store/normalized_service_case_feedback.csv
-> Quarantined: 0 rows -> /content/data/quarantine_queue/quarantine_service_case_feedback.csv

Processing file: curated_correspondence.csv
-> Accepted: 11426 rows -> /content/data/normalized_store/normalized_curated_correspondence.csv
-> Quarantined: 0 rows -> /content/data/quarantine_queue/quarantine_curated_correspondence.csv

Processing file: survey_responses.csv
-> Accepted: 300 rows -> /content/data/normalized_store/normalized_survey_responses.csv
-> Quarantined: 0 rows -> /content/data/quarantine_queue/quarantine_survey_responses.csv

================ BATCH RECONCILIATION SUMMARY (DAT 02) ================


,source_file,rows_received,rows_accepted,rows_quarantined,status
0,service_case_feedback.csv,300,300,0,VERIFIED
1,curated_correspondence.csv,11426,11426,0,VERIFIED
2,survey_responses.csv,300,300,0,VERIFIED


## 5. Nén và Tải Dữ liệu Đầu ra về Máy Cục bộ từ Google Colab

In [ ]:
# Nén toàn bộ dữ liệu normalized_store và raw_intake thành file zip để tải về
!zip -r /content/songnhat_intake_output.zip /content/data

try:
    from google.colab import files
    print("Triggering browser download for songnhat_intake_output.zip...")
    files.download("/content/songnhat_intake_output.zip")
except Exception as e:
    print("Download notice (run manually if not in Colab UI):", e)

	zip warning: name not matched: /content/data

zip error: Nothing to do! (try: zip -r /content/songnhat_intake_output.zip . -i /content/data)
Triggering browser download for songnhat_intake_output.zip...
Download notice (run manually if not in Colab UI): Cannot find file: /content/songnhat_intake_output.zip
